# 📊 01 — Raw Data Exploration
This notebook performs Exploratory Data Analysis (EDA) on the RDD-2022 and BharatPotHole datasets.
All generated charts are saved to the `notebooks/eda_outputs` directory.



In [ ]:
import os
import sys
import glob
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

# Configuration
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
sns.set_palette('husl')

# Project Paths
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'

RDD_ROOT = RAW_DATA_DIR / 'rdd2022' / 'RDD_SPLIT'
BHARAT_ROOT = RAW_DATA_DIR / 'bharatpothole' / 'BharatPotHole' / 'BharatPotHole'

RDD_CLASSES = {0: 'Longitudinal Crack', 1: 'Transverse Crack', 2: 'Alligator Crack', 3: 'Other Corruption', 4: 'Pothole'}
BHARAT_CLASSES = {0: 'Pothole'}

OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'eda_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



## 📂 2. Directory Structure


In [ ]:
def explore_directory(root_path, dataset_name, max_depth=3):
    print(f"\n🗂️  {dataset_name}")
    print(f"   Root: {root_path}\n   Exists: {root_path.exists()}")
    if not root_path.exists(): return
    for dirpath, dirnames, filenames in os.walk(root_path):
        depth = len(Path(dirpath).relative_to(root_path).parts)
        if depth >= max_depth: continue
        indent = "   " + "│  " * depth
        print(f"{indent}├── 📁 {os.path.basename(dirpath)}/ ({len(dirnames)} dirs, {len(filenames)} files)")

explore_directory(RDD_ROOT, "RDD-2022")
explore_directory(BHARAT_ROOT, "BharatPotHole")



## 📊 3. Image and Label Counts


In [ ]:
def get_dataset_stats(root_path, splits, dataset_name):
    stats = {}
    for split in splits:
        img_dir, lbl_dir = root_path / split / 'images', root_path / split / 'labels'
        n_imgs = sum(1 for ext in ['.jpg', '.jpeg', '.png'] if img_dir.exists() for _ in img_dir.glob(f'*{ext}'))
        n_lbls = sum(1 for _ in lbl_dir.glob('*.txt')) if lbl_dir.exists() else 0
        stats[split] = {'images': n_imgs, 'labels': n_lbls, 'img_dir': img_dir, 'lbl_dir': lbl_dir}
        print(f"\n   📁 {dataset_name} / {split}:\n      🖼️  Images: {n_imgs:,}\n      🏷️  Labels: {n_lbls:,}")
        if n_imgs != n_lbls: print(f"      ⚠️  Difference: abs({n_imgs - n_lbls}) files")
    return stats

print("\n🔵 RDD-2022:")
rdd_stats = get_dataset_stats(RDD_ROOT, ['train', 'val', 'test'], 'RDD-2022')
print("\n🟢 BharatPotHole:")
bharat_stats = get_dataset_stats(BHARAT_ROOT, ['train', 'valid', 'test'], 'BharatPotHole')



## 📊 4. Plot Split Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

def plot_counts(ax, stats, title, color_img, color_lbl):
    splits = list(stats.keys())
    imgs = [stats[s]['images'] for s in splits]
    lbls = [stats[s]['labels'] for s in splits]
    x, width = np.arange(len(splits)), 0.35
    b1 = ax.bar(x - width/2, imgs, width, label='Images', color=color_img, edgecolor='white')
    b2 = ax.bar(x + width/2, lbls, width, label='Labels', color=color_lbl, edgecolor='white')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels([s.upper() for s in splits])
    ax.legend(); ax.set_ylabel('Count')
    for bars in [b1, b2]:
        for b in bars: ax.text(b.get_x() + b.get_width()/2., b.get_height() + 50, f'{int(b.get_height()):,}', ha='center', va='bottom', fontsize=10)

plot_counts(axes[0], rdd_stats, '🔵 RDD-2022 — Image/Label count per split', '#3498db', '#e74c3c')
plot_counts(axes[1], bharat_stats, '🟢 BharatPotHole — Image/Label count per split', '#2ecc71', '#f39c12')

plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'dataset_split_counts.png', dpi=150); plt.show()



## 📐 5. Image Size Analysis


In [ ]:
def analyze_image_sizes(img_dir, dataset_name, max_samples=2000):
    img_files = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    if len(img_files) > max_samples: img_files = random.sample(img_files, max_samples)
    widths, heights, aspects, file_sizes = [], [], [], []
    for f in tqdm(img_files, desc=f"Reading {dataset_name}", leave=False):
        try:
            with Image.open(f) as img:
                w, h = img.size
                widths.append(w); heights.append(h); aspects.append(w/h); file_sizes.append(f.stat().st_size / 1024)
        except Exception: pass
    return pd.DataFrame({'width': widths, 'height': heights, 'aspect_ratio': aspects, 'file_size_kb': file_sizes, 'dataset': dataset_name})

all_img_stats = pd.concat([
    analyze_image_sizes(RDD_ROOT/'train'/'images', 'RDD-2022'), 
    analyze_image_sizes(BHARAT_ROOT/'train'/'images', 'BharatPotHole')
], ignore_index=True)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ds, c in [('RDD-2022', '#3498db'), ('BharatPotHole', '#2ecc71')]:
    sub = all_img_stats[all_img_stats['dataset'] == ds]
    axes[0, 0].hist(sub['width'], bins=50, alpha=0.7, label=ds, color=c)
    axes[0, 1].hist(sub['height'], bins=50, alpha=0.7, label=ds, color=c)
    axes[1, 0].hist(sub['aspect_ratio'], bins=50, alpha=0.7, label=ds, color=c)
    axes[1, 1].scatter(sub['width'], sub['height'], alpha=0.3, label=ds, color=c, s=10)

axes[0,0].set_title('Width Distribution'); axes[0,1].set_title('Height Distribution')
axes[1,0].set_title('Aspect Ratio (W/H)'); axes[1,1].set_title('Width vs Height')
for ax in axes.flat: ax.legend()
plt.suptitle('📐 Image Size Analysis', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'image_size_analysis.png', dpi=150); plt.show()



## 🏷️ 6. Labels Analysis (YOLO Format)


In [ ]:
def parse_yolo_labels(label_dir, class_names):
    annotations = []
    label_files = list(label_dir.glob('*.txt'))
    for lbl_file in tqdm(label_files, desc="Parsing labels", leave=False):
        lines = [l.strip() for l in open(lbl_file).readlines() if l.strip()]
        if not lines:
            annotations.append({'file': lbl_file.stem, 'class_id': -1, 'class_name': 'NO_ANNOTATION', 'x_center': 0, 'y_center': 0, 'bbox_width': 0, 'bbox_height': 0, 'n_objects': 0})
            continue
        for line in lines:
            parts = line.split()
            if len(parts) >= 5:
                cid = int(parts[0])
                annotations.append({'file': lbl_file.stem, 'class_id': cid, 'class_name': class_names.get(cid, f'Unknown_{cid}'),
                                    'x_center': float(parts[1]), 'y_center': float(parts[2]), 'bbox_width': float(parts[3]), 'bbox_height': float(parts[4]), 'n_objects': len(lines)})
    return pd.DataFrame(annotations)

rdd_labels = parse_yolo_labels(RDD_ROOT / 'train' / 'labels', RDD_CLASSES)
bharat_labels = parse_yolo_labels(BHARAT_ROOT / 'train' / 'labels', BHARAT_CLASSES)

rdd_valid = rdd_labels[rdd_labels['class_id'] >= 0]
bharat_valid = bharat_labels[bharat_labels['class_id'] >= 0]

print(f"🔵 RDD-2022: {len(rdd_valid):,} valid annotations in {len(rdd_labels.file.unique()):,} files.")
print(f"🟢 BharatPotHole: {len(bharat_valid):,} valid annotations in {len(bharat_labels.file.unique()):,} files.")



## 📊 7. Class Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

rdd_counts = rdd_valid['class_name'].value_counts()
axes[0].pie(rdd_counts, labels=rdd_counts.index, autopct='%1.1f%%', colors=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#DDA0DD'], startangle=90)
axes[0].set_title('🔵 RDD-2022 — Class Distribution', fontsize=14, fontweight='bold')

bharat_counts = bharat_valid['class_name'].value_counts()
bars = axes[1].barh(bharat_counts.index, bharat_counts.values, color='#2ecc71', height=0.5)
axes[1].set_title('🟢 BharatPotHole — Class Distribution', fontsize=14, fontweight='bold')
for bar in bars: axes[1].text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2., f'{int(bar.get_width()):,}', va='center')

plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'class_distribution.png', dpi=150); plt.show()



## 📊 8. Objects per Image


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, df, title, color in zip(axes, [rdd_valid, bharat_valid], ['🔵 RDD-2022 — Objects/Image', '🟢 BharatPotHole — Objects/Image'], ['#3498db', '#2ecc71']):
    obj_counts = df.groupby('file').size()
    ax.hist(obj_counts, bins=range(1, obj_counts.max() + 2), color=color, rwidth=0.85)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axvline(obj_counts.mean(), color='red', linestyle='--', label=f'Mean: {obj_counts.mean():.2f}')
    ax.legend(); ax.set_xlabel('Objects'); ax.set_ylabel('Images')

plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'objects_per_image.png', dpi=150); plt.show()



## 📊 9. Bounding Box Sizes & Areas


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].scatter(rdd_valid['bbox_width'], rdd_valid['bbox_height'], c=rdd_valid['class_id'], cmap='Set1', alpha=0.3, s=8)
axes[0, 1].scatter(bharat_valid['bbox_width'], bharat_valid['bbox_height'], color='#2ecc71', alpha=0.3, s=8)
axes[0,0].set_title('🔵 RDD-2022 — BBox Width vs Height'); axes[0,1].set_title('🟢 BharatPotHole — BBox Width vs Height')

rdd_area = rdd_valid['bbox_width'] * rdd_valid['bbox_height']
bharat_area = bharat_valid['bbox_width'] * bharat_valid['bbox_height']

axes[1, 0].hist(rdd_area, bins=100, color='#3498db'); axes[1, 0].set_title('🔵 RDD-2022 — Area Distribution')
axes[1, 1].hist(bharat_area, bins=100, color='#2ecc71'); axes[1, 1].set_title('🟢 BharatPotHole — Area Distribution')

plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'bbox_size_analysis.png', dpi=150); plt.show()



## 🗺️ 10. Bounding Box Center Heatmap


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

im1 = axes[0].hist2d(rdd_valid['x_center'], rdd_valid['y_center'], bins=50, cmap='YlOrRd', range=[[0, 1], [0, 1]])[3]
axes[0].set_title('🔵 RDD-2022 — Center Heatmap'); axes[0].invert_yaxis(); plt.colorbar(im1, ax=axes[0])

im2 = axes[1].hist2d(bharat_valid['x_center'], bharat_valid['y_center'], bins=50, cmap='YlGn', range=[[0, 1], [0, 1]])[3]
axes[1].set_title('🟢 BharatPotHole — Center Heatmap'); axes[1].invert_yaxis(); plt.colorbar(im2, ax=axes[1])

plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'bbox_center_heatmap.png', dpi=150); plt.show()



## 🖼️ 11. Sample Images with Bounding Boxes


In [ ]:
CLASS_COLORS = {0: '#FF6B6B', 1: '#4ECDC4', 2: '#45B7D1', 3: '#96CEB4', 4: '#DDA0DD'}

def show_sample_images(img_dir, lbl_dir, class_names, dataset_name, n_samples=8, n_cols=4):
    img_files = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    labeled_files = [f for f in img_files if (lbl_dir / (f.stem + '.txt')).exists() and (lbl_dir / (f.stem + '.txt')).stat().st_size > 0]
    selected = random.sample(labeled_files, min(n_samples, len(labeled_files)))
    
    n_rows = (len(selected) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    if n_rows == 1: axes = axes.reshape(1, -1)
    
    for idx, img_f in enumerate(selected):
        ax = axes[idx // n_cols, idx % n_cols]
        img = Image.open(img_f)
        ax.imshow(img); ax.axis('off'); ax.set_title(img_f.name[:30])
        
        lines = [l.strip() for l in open(lbl_dir / (img_f.stem + '.txt')).readlines() if l.strip()]
        for line in lines:
            parts = line.split()
            if len(parts) >= 5:
                cid, x, y, w, h = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1, y1 = (x - w/2) * img.width, (y - h/2) * img.height
                c = CLASS_COLORS.get(cid, '#FFFFFF')
                rect = patches.Rectangle((x1, y1), w*img.width, h*img.height, linewidth=2, edgecolor=c, facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1 - 5, class_names.get(cid, 'Unk'), color='white', fontsize=8, bbox=dict(facecolor=c, edgecolor='none', pad=0.2))
                
    for idx in range(len(selected), n_rows * n_cols): axes[idx // n_cols, idx % n_cols].axis('off')
    plt.suptitle(f'🖼️ {dataset_name} — Sample Annotations', fontsize=16, fontweight='bold')
    plt.tight_layout(); plt.savefig(OUTPUT_DIR / f'samples_{dataset_name}.png', dpi=150); plt.show()

show_sample_images(RDD_ROOT / 'train' / 'images', RDD_ROOT / 'train' / 'labels', RDD_CLASSES, 'RDD-2022')
show_sample_images(BHARAT_ROOT / 'train' / 'images', BHARAT_ROOT / 'train' / 'labels', BHARAT_CLASSES, 'BharatPotHole')



## 📋 12. Conclusion
- Both datasets have been parsed. 
- BharatPotHole is purely focused on potholes (1 class).
- RDD-2022 has a diverse set of road damages (5 classes), with cracks dominating.
- BBox distributions vary widely, confirming the need to handle multi-scale objects during model training.

